# Synthetic CFD-like Signal Forecasting with TensorFlow/Keras

This notebook works with fully turbulent signal.

- several time scales,
- oscillations,
- random noise,
- intermittent burst-like events.


```text
signal → input windows → future prediction → model comparison
```

The learning task is:

```text
Input:  previous values of u(t)
Output: future value of u(t)
```

We compare several methods:

```text
1. Persistence baseline
2. Dense baseline
3. SimpleRNN
4. GRU
5. LSTM
```

We also use **EarlyStopping** to stop training automatically when the validation loss stops improving.

# Stage 1 — Extracting one near-wall velocity signal from 106 simulation files

**Portfolio project: forecasting a turbulent channel-flow signal with recurrent networks and a Transformer.**

This notebook turns 106 raw output files of a direct numerical simulation of turbulent channel flow (`P125_83pi_30001.stp` … `30106.stp`, HDF5, 1,000 time steps each, 5.3 GB in total) into one clean time series: the streamwise velocity `utmp` at the wall-normal index 50 (y = −0.6196), 106,000 samples, saved as one 0.95 MB HDF5 file with the file index of every sample, so that later windows never straddle two files. It also writes the same signal as 106 TFRecord shards. Everything is done by the class `CFDTimeSeriesLoader` in `cfd_data_tools.py`, which the next stages import to cut the signal into windows, split it in time and normalise it.

**How it connects.** The file it writes, `data/utmp_y50_raw_timeseries.h5`, is the input of Stage 2 (recurrent networks), Stage 3 (Transformer) and Stage 4 (horizon and baselines). The copy in `data/` is byte-identical to the one this notebook writes (SHA-1 `f0a7789027c49e6465af62148cfbc1ea2e6aa22e`), so Stages 2–4 run without the raw files.

**About the outputs.** The outputs below come from my single run of this notebook (Apple M3 Max, 2026-10-13). Afterwards I changed only text for this repository: this cell and the paths cell below, the path lines (raw files from `../release_assets/raw_stp/`, the processed file to `../data/`, the TFRecords to `../results/01_extract_cfd_signal/`), comments and printed paths. The original title cell below still says "Synthetic CFD-like Signal Forecasting"; the notebook reads real simulation files and trains no model.

In [ ]:
# Paths in the repository layout (run this notebook from notebooks/)
from pathlib import Path
RAW_DIR = Path("../release_assets/raw_stp")              # the 106 .stp files, from the release archives raw_stp_part1-3.tar
DATA_DIR = Path("../data")                               # the processed signal (also in git)
RESULTS = Path("../results/01_extract_cfd_signal")      # the TFRecord shards
RESULTS.mkdir(parents=True, exist_ok=True)
print("raw files present:", RAW_DIR.is_dir() and len(list(RAW_DIR.glob("*.stp"))))

In [1]:
# Device report -- CPU exception: the Apple GPU is hidden from TensorFlow for this notebook (reason and failing op in RUN.md)
import tensorflow as tf, platform
tf.config.set_visible_devices([], "GPU")
print("TensorFlow", tf.__version__, "| Python", platform.python_version())
print("Physical devices:", tf.config.list_physical_devices(), "| visible to TensorFlow:", tf.config.get_visible_devices())

TensorFlow 2.18.1 | Python 3.12.14
Physical devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')] | visible to TensorFlow: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


# Environment

This notebook should run in:

- Google Colab,
- VS Code with Jupyter,
- a local Jupyter installation.

Required packages:

```bash
pip install numpy matplotlib tensorflow ipykernel
```

# Loading and Preparing a CFD Time Signal

In this section, we use the helper class `CFDTimeSeriesLoader` to automate the first steps of the CFD data workflow.

The goal is to read several CFD files, extract one time signal from a selected wall-normal location, concatenate the signals, and save the processed result in a clean HDF5 file.

The complete workflow is:

```text
raw CFD files
    ↓
open HDF5/STP files
    ↓
read time, y coordinate, and selected variable
    ↓
extract one signal at a fixed y index
    ↓
concatenate all files into one long time series
    ↓
save the processed time series as HDF5
```

We first import the class:

```python
from cfd_data_tools import CFDTimeSeriesLoader
```

This class contains several utility functions for reading CFD data and preparing it for machine learning.

---

## Creating the Loader

We create an instance of the class:

```python
loader = CFDTimeSeriesLoader(
    data_dir="data",
    dataset_name="sta/utmp",
    time_name="header/timev",
    y_name="header/y",
    y_index=50,
    variable="utmp"
)
```

The arguments define where the data are stored and which quantities we want to read.

```text
data_dir      → folder containing the CFD files
dataset_name  → internal HDF5 path of the variable to read
time_name     → internal HDF5 path of the time vector
y_name        → internal HDF5 path of the wall-normal coordinate
y_index       → selected wall-normal index
variable      → name used when saving the processed file
```

In this example, we read:

```text
sta/utmp
```

which is assumed to contain a time-dependent signal with shape:

```text
number of time samples × number of y points
```

Then:

```python
y_index = 50
```

means that we extract the signal at the wall-normal position:

```python
y[50]
```

---

## Checking the Environment

```python
loader.print_environment()
```

This prints useful information such as:

```text
operating system
whether we are running in Google Colab
current working directory
selected data directory
```

This is useful because path errors are one of the most common problems when reading data.

---

## Listing the Source Files

```python
loader.list_source_files()
```

This searches the `data` folder for CFD files with the selected extension.

By default, the class looks for files such as:

```text
*.stp
```

The method prints how many files were found and shows the first few filenames.

---

## Extracting the Time Signal

```python
loader.extract_signal_from_raw_files()
```

This is the main reading step.

For each file, the class opens the HDF5/STP file and reads:

```python
timev = f["header/timev"][:]
y = f["header/y"][:]
signal = f["sta/utmp"][:, y_index]
```

The result from each file is appended to a list. After all files have been read, the signals are concatenated into one long time series:

```text
all_timev
all_signal
all_file_indices
sample_index
```

The array `all_file_indices` is important because it tells us which original file each sample came from. This allows us later to create machine learning windows without crossing file boundaries.

---

## Saving the Processed Data

```python
loader.save_processed_h5()
```

This saves the processed time series into a new HDF5 file.

The saved file contains:

```text
raw/timev
raw/signal
raw/file_indices
raw/sample_index
files/names
files/lengths
files/start_indices
y
```

It also stores metadata as HDF5 attributes, for example:

```text
variable
original dataset name
time dataset name
y dataset name
selected y index
selected y value
number of files
number of total samples
```

This processed HDF5 file is much easier to use later for machine learning.

Instead of repeatedly opening many raw CFD files, we can directly load one clean processed file.

---

## Key Idea

This cell converts many raw CFD files into one machine-learning-ready time-series file.

```text
many raw CFD files → one clean processed HDF5 file
```

After this step, we can create:

```text
training windows
validation windows
test windows
TensorFlow datasets
TFRecords
RNN / GRU / LSTM / Transformer inputs
β-VAE training windows
```

In [2]:
!ls /content/drive/<Drive folder>/cfd-signal-forecasting-rnn-transformer/

ls: /content/drive/<Drive folder>/cfd-signal-forecasting-rnn-transformer/: No such file or directory


In [3]:
# =========================================================
# Google Colab: mount Drive and move to project folder
# =========================================================

from pathlib import Path
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    project_dir = Path('/content/drive/<Drive folder>/cfd-signal-forecasting-rnn-transformer/notebooks')
    data=Path('/content/drive/<Drive folder>/cfd-signal-forecasting-rnn-transformer/release_assets/raw_stp')

    %cd '/content/drive/<Drive folder>/cfd-signal-forecasting-rnn-transformer/notebooks'

    if str(project_dir) not in sys.path:
        sys.path.append(str(project_dir))

    print("Project directory:")
    print(project_dir)

    print("Data directory:")
    print(data)

    print("\nFiles:")
    for item in project_dir.iterdir():
        print(" -", item.name)

else:
    print("This is not Google Colab.")

This is not Google Colab.


In [4]:
# =========================================================
# Reload local library after editing
# =========================================================

import importlib
import cfd_data_tools # type: ignore

importlib.reload(cfd_data_tools)

from cfd_data_tools import CFDTimeSeriesLoader, H5ToTFRecordConverter # type: ignore

print("cfd_data_tools reloaded successfully.")

cfd_data_tools reloaded successfully.


In [5]:
# =========================================================
# Data path and start the loader
# =========================================================


loader = CFDTimeSeriesLoader(
    # [original] data_dir=data,
    data_dir=Path("../release_assets/raw_stp"),  # modified path: `data` is defined only in the Colab branch above
    processed_dir="../data",  # path line: the processed file goes to data/
    dataset_name="sta/utmp",
    time_name="header/timev",
    y_name="header/y",
    y_index=50,
    variable="utmp"
)

loader.print_environment()
loader.list_source_files()

loader.extract_signal_from_raw_files()
loader.save_processed_h5()

Operating system: Darwin
Running in Google Colab: False
Current working directory: <repository>/notebooks
Selected data directory: ../release_assets/raw_stp

Data directory found.
Data directory: ../release_assets/raw_stp
Number of .stp files found: 106
 - P125_83pi_30001.stp
 - P125_83pi_30002.stp
 - P125_83pi_30003.stp
 - P125_83pi_30004.stp
 - P125_83pi_30005.stp
 - P125_83pi_30006.stp
 - P125_83pi_30007.stp
 - P125_83pi_30008.stp
 - P125_83pi_30009.stp
 - P125_83pi_30010.stp
 ...
Read file 0: P125_83pi_30001.stp, samples: 1000
Read file 1: P125_83pi_30002.stp, samples: 1000
Read file 2: P125_83pi_30003.stp, samples: 1000
Read file 3: P125_83pi_30004.stp, samples: 1000
Read file 4: P125_83pi_30005.stp, samples: 1000
Read file 5: P125_83pi_30006.stp, samples: 1000
Read file 6: P125_83pi_30007.stp, samples: 1000
Read file 7: P125_83pi_30008.stp, samples: 1000
Read file 8: P125_83pi_30009.stp, samples: 1000
Read file 9: P125_83pi_30010.stp, samples: 1000
Read file 10: P125_83pi_30011.s

Read file 40: P125_83pi_30041.stp, samples: 1000
Read file 41: P125_83pi_30042.stp, samples: 1000
Read file 42: P125_83pi_30043.stp, samples: 1000
Read file 43: P125_83pi_30044.stp, samples: 1000
Read file 44: P125_83pi_30045.stp, samples: 1000
Read file 45: P125_83pi_30046.stp, samples: 1000
Read file 46: P125_83pi_30047.stp, samples: 1000
Read file 47: P125_83pi_30048.stp, samples: 1000
Read file 48: P125_83pi_30049.stp, samples: 1000
Read file 49: P125_83pi_30050.stp, samples: 1000
Read file 50: P125_83pi_30051.stp, samples: 1000
Read file 51: P125_83pi_30052.stp, samples: 1000
Read file 52: P125_83pi_30053.stp, samples: 1000
Read file 53: P125_83pi_30054.stp, samples: 1000
Read file 54: P125_83pi_30055.stp, samples: 1000
Read file 55: P125_83pi_30056.stp, samples: 1000
Read file 56: P125_83pi_30057.stp, samples: 1000
Read file 57: P125_83pi_30058.stp, samples: 1000
Read file 58: P125_83pi_30059.stp, samples: 1000
Read file 59: P125_83pi_30060.stp, samples: 1000
Read file 60: P125_8

Processed HDF5 saved to:
../data/utmp_y50_raw_timeseries.h5


PosixPath('../data/utmp_y50_raw_timeseries.h5')

# Preparing Forecasting Data from a Processed HDF5 File

In the previous step, we converted several raw CFD files into one clean processed HDF5 file.

Now we use that processed file to prepare the data for a machine learning forecasting problem.

The goal is to create a dataset of the form:

```text
past signal window → future signal window
```

For example:

```text
300 previous values → 10 future values
```

This type of dataset can be used to train models such as:

```text
Dense networks
RNNs
GRUs
LSTMs
Conv1D models
Transformers
```

---

## Creating the Loader

We first import the helper class:

```python
from cfd_data_tools import CFDTimeSeriesLoader
```

Then we create a loader object:

```python
loader = CFDTimeSeriesLoader(
    processed_dir="../data",
    y_index=50,
    variable="utmp"
)
```

In this case, we are not reading the original raw CFD files again.

Instead, we are going to read a processed HDF5 file that was already created before.

```text
../data/utmp_y50_raw_timeseries.h5
```

This file already contains the concatenated time signal, the time vector, the file indices, and useful metadata.

---

## Loading the Processed HDF5 File

```python
loader.load_processed_h5("../data/utmp_y50_raw_timeseries.h5")
```

This loads the processed time series into the loader.

Internally, the loader reads arrays such as:

```text
raw/timev
raw/signal
raw/file_indices
raw/sample_index
```

The most important array for forecasting is:

```text
raw/signal
```

which contains the full time signal extracted from the CFD data.

The array:

```text
raw/file_indices
```

is also important because it tells us which original CFD file each time sample came from. This allows us to avoid creating input windows that cross from one file to another.

---

## Creating Forecasting Windows

The main step is:

```python
data = loader.prepare_forecasting_data(
    input_window=300,
    output_window=10,
    stride=10,
    batch_size=64
)
```

This function performs several operations automatically.

The input window is:

```python
input_window = 300
```

This means that each input sample contains 300 consecutive values of the signal.

The output window is:

```python
output_window = 10
```

This means that the target contains the next 10 values after the input window.

The stride is:

```python
stride = 10
```

This means that we move the window 10 time steps forward to create the next training sample.

Schematically:

```text
X sample 1: u(0),   u(1),   ..., u(299)
y sample 1: u(300), u(301), ..., u(309)

X sample 2: u(10),  u(11),  ..., u(309)
y sample 2: u(310), u(311), ..., u(319)

X sample 3: u(20),  u(21),  ..., u(319)
y sample 3: u(320), u(321), ..., u(329)
```

Therefore, the machine learning problem is:

```text
given the previous 300 values, predict the next 10 values
```

---

## Train, Validation, and Test Split

The function also splits the windows chronologically into:

```text
training data
validation data
test data
```

A chronological split is important for time series.

We do not randomly mix the full time signal before splitting because that would allow information from the future to leak into the training set.

The typical structure is:

```text
early part of the signal      → training
middle part of the signal     → validation
final part of the signal      → test
```

---

## Normalization

The function also normalizes the data using only the training statistics.

It computes:

```python
signal_mean = X_train.mean()
signal_std = X_train.std()
```

and then applies:

```python
X_norm = (X - signal_mean) / signal_std
```

The same mean and standard deviation are used for training, validation, and test data.

This is important because the validation and test sets must not influence the normalization.

In other words:

```text
training data defines the scale
validation and test data use that same scale
```

---

## TensorFlow Datasets

The function also creates TensorFlow datasets:

```python
train_ds = data["train_ds"]
val_ds = data["val_ds"]
test_ds = data["test_ds"]
```

These objects can be passed directly to Keras:

```python
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=...
)
```

The datasets are already batched:

```python
batch_size = 64
```

This means that the model receives 64 samples at a time during training.

---

## Accessing the NumPy Arrays

We can also access the normalized NumPy arrays directly:

```python
X_train = data["X_train"]
y_train = data["y_train"]
```

For this example, the expected shapes are:

```text
X_train.shape → (number of training samples, 300, 1)
y_train.shape → (number of training samples, 10)
```

The last dimension in `X_train` is the number of input channels.

Since we are using only one signal, the number of channels is:

```text
1
```

So each input sample has shape:

```text
300 × 1
```

and each target has shape:

```text
10
```

---

## Key Idea

This cell transforms a processed CFD time signal into a complete machine learning dataset.

```text
processed HDF5 file
    ↓
load time signal
    ↓
create input/output windows
    ↓
split into train, validation, and test
    ↓
normalize using training statistics only
    ↓
create TensorFlow datasets
```

After this step, the data are ready to train forecasting models.

# Converting the Processed HDF5 File to TFRecords

In the previous section, we created a processed HDF5 file containing a clean CFD time series.

Now we convert that processed HDF5 file into TFRecord files.

TFRecords are a TensorFlow-oriented binary format designed to store data efficiently for machine learning workflows.

The workflow is:

```text
processed HDF5 file
    ↓
read time signal and metadata
    ↓
split data by original CFD file segment
    ↓
write TFRecord files
    ↓
save metadata describing the conversion
```

---

## Why Convert HDF5 to TFRecords?

HDF5 is an excellent scientific format.

It is useful for:

```text
storing CFD data
organizing groups and datasets
saving coordinates and metadata
post-processing
visualization
scientific inspection
```

However, TensorFlow models usually train more efficiently when the data are provided through a `tf.data` pipeline.

TFRecords are useful because they allow TensorFlow to stream data efficiently during training.

A typical TensorFlow workflow is:

```text
TFRecord files
    ↓
TFRecordDataset
    ↓
parse records
    ↓
shuffle
    ↓
batch
    ↓
prefetch
    ↓
model.fit()
```

The key idea is:

```text
HDF5 is convenient for scientific storage.
TFRecord is convenient for TensorFlow training.
```

---

## Importing the Converter Class

We first import the class:

```python
from cfd_data_tools import H5ToTFRecordConverter
```

This class reads a processed HDF5 file and writes the corresponding TFRecord files.

---

## Creating the Converter

We create the converter object:

```python
converter = H5ToTFRecordConverter(
    h5_file="../data/utmp_y50_raw_timeseries.h5",
    output_dir="../results/01_extract_cfd_signal/utmp_y50_tfrecords",
    compression_type=""
)
```

The arguments are:

```text
h5_file           → input processed HDF5 file
output_dir        → folder where the TFRecord files will be saved
compression_type  → compression option
```

Here:

```python
compression_type=""
```

means that the TFRecord files are saved without compression.

If we wanted gzip compression, we could use:

```python
compression_type="GZIP"
```

Compression can reduce file size, but reading may become slightly slower.

---

## Writing Raw TFRecords

The command:

```python
converter.write_raw_tfrecords()
```

reads the processed HDF5 file and writes TFRecord files.

In this version, we write **raw time-series TFRecords**.

That means each TFRecord stores a full segment of the time signal corresponding to one original CFD file.

Schematically:

```text
original CFD file 1 → raw_00000.tfrecord
original CFD file 2 → raw_00001.tfrecord
original CFD file 3 → raw_00002.tfrecord
...
```

Each TFRecord contains information such as:

```text
file_id
file_name
start_index
length
timev
signal
file_indices
sample_index
```

This preserves the connection between the processed time series and the original CFD files.

---

## Difference Between Raw TFRecords and Window TFRecords

There are two possible TFRecord strategies.

### Raw TFRecords

```text
one record → one complete signal segment
```

This is useful when we want to keep the data close to the original structure.

For example:

```text
record 1 → all time samples from file 1
record 2 → all time samples from file 2
record 3 → all time samples from file 3
```

Later, we can still decide how to create training windows.

---

### Window TFRecords

```text
one record → one machine learning sample
```

This is useful when the training examples are already fixed.

For example:

```text
record 1 → X window, y window
record 2 → X window, y window
record 3 → X window, y window
```

where:

```text
X = past 300 values
y = next 10 values
```

This is usually more efficient for direct TensorFlow training.

---

## Key Idea

This cell converts the processed CFD time series into a TensorFlow-friendly format.

```text
processed HDF5
    ↓
TFRecords
```

The processed HDF5 file remains the main scientific file.

The TFRecord files are an additional machine learning format that can be used later to train TensorFlow models more efficiently.

In practice, a good workflow is:

```text
raw CFD files
    ↓
processed HDF5
    ↓
TFRecords
    ↓
tf.data.Dataset
    ↓
Keras model
```

In [6]:
from cfd_data_tools import H5ToTFRecordConverter # type: ignore

converter = H5ToTFRecordConverter(
    h5_file="../data/utmp_y50_raw_timeseries.h5",
    output_dir="../results/01_extract_cfd_signal/utmp_y50_tfrecords",
    compression_type=""
)

raw_data=True

if raw_data:
    converter.write_raw_tfrecords()
else:
    converter.write_window_tfrecords(
        input_window=300,
        output_window=10,
        stride=10,
        records_per_shard=512
)

Saved: raw_00000.tfrecord
Saved: raw_00001.tfrecord
Saved: raw_00002.tfrecord
Saved: raw_00003.tfrecord
Saved: raw_00004.tfrecord
Saved: raw_00005.tfrecord
Saved: raw_00006.tfrecord
Saved: raw_00007.tfrecord
Saved: raw_00008.tfrecord
Saved: raw_00009.tfrecord
Saved: raw_00010.tfrecord
Saved: raw_00011.tfrecord
Saved: raw_00012.tfrecord
Saved: raw_00013.tfrecord
Saved: raw_00014.tfrecord
Saved: raw_00015.tfrecord
Saved: raw_00016.tfrecord
Saved: raw_00017.tfrecord
Saved: raw_00018.tfrecord
Saved: raw_00019.tfrecord
Saved: raw_00020.tfrecord
Saved: raw_00021.tfrecord
Saved: raw_00022.tfrecord
Saved: raw_00023.tfrecord
Saved: raw_00024.tfrecord
Saved: raw_00025.tfrecord
Saved: raw_00026.tfrecord
Saved: raw_00027.tfrecord
Saved: raw_00028.tfrecord
Saved: raw_00029.tfrecord
Saved: raw_00030.tfrecord
Saved: raw_00031.tfrecord
Saved: raw_00032.tfrecord
Saved: raw_00033.tfrecord
Saved: raw_00034.tfrecord
Saved: raw_00035.tfrecord
Saved: raw_00036.tfrecord
Saved: raw_00037.tfrecord
Saved: raw_0